# UC-UW-2 — Upload an Asset + DW JSONLines via GcsDetailedReporter

**Goal:** Use the catalog's GCP-plugin bucket as the simulated DWH landing zone. Upload an asset, drive an ingestion process with `GcsDetailedReporter` writing JSONLines under `gs://{catalog_bucket}/{collection_id}/reports/`, and demonstrate the asset-id dedup refusal flowing into the report.

**Pub/sub note (REMOTE_ONLY):** the asset row materializes asynchronously via an `OBJECT_FINALIZE` Pub/Sub event. Local stacks cannot receive the push callback (no `K_SERVICE`). Step 4 detects this and either:
- **remote:** waits for the pub/sub-driven asset auto-creation, then runs ingestion
- **local:** registers the asset manually (so the rest of the notebook still exercises the reporter end-to-end)

**Prereq:** notebook 01 ran (catalog + collection exist, write policy + geometry stats configured). Either rerun 01 first or set `RUN_ID` to reuse.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"uw_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

# REMOTE_ONLY heuristic: pub/sub-driven flows do not fire against localhost.
IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL
PUBSUB_ENABLED = not IS_LOCAL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"RUN_ID        : {RUN_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
print(f"PUBSUB        : {'enabled (remote)' if PUBSUB_ENABLED else 'disabled (local)'}")


## Step 0 — Discover the catalog's bucket

In [ ]:
# Step 0 — Discover the catalog's bucket (GCP plugin).
r = client.get(f"/gcp/buckets/catalogs/{CATALOG_ID}")
if r.status_code != 200:
    raise RuntimeError(
        f"Catalog bucket lookup failed: {r.status_code}: {r.text[:200]}\n"
        f"  → ensure GCP plugin is enabled and the catalog was created via notebook 01."
    )
bucket_info = r.json()
BUCKET = bucket_info.get("bucket_id") or bucket_info.get("bucket_name")
assert BUCKET, f"no bucket name in response: {bucket_info}"
print(f"CATALOG_BUCKET : gs://{BUCKET}")

REPORT_PREFIX = f"gs://{BUCKET}/{COLLECTION_ID}/reports"
print(f"REPORT_PREFIX  : {REPORT_PREFIX}")


## Step 1 — init-upload (signed resumable PUT URL)

In [ ]:
# Step 1 — init-upload (signed resumable PUT URL)
ASSET_ID = f"asset_{RUN_ID}"
FILENAME = f"{ASSET_ID}.geojson"
CONTENT_TYPE = "application/geo+json"

init_payload = {
    "filename": FILENAME,
    "content_type": CONTENT_TYPE,
    "asset": {
        "asset_id": ASSET_ID,
        "asset_type": "VECTORIAL",
        "metadata": {"source": "ui-walkthrough-nb02"},
    },
    "upload_options": {"predefined_acl": "publicRead", "if_generation_match": 0},
}

r = client.post(
    f"/gcp/buckets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/init-upload",
    content=json.dumps(init_payload),
)
print(f"init-upload: {r.status_code}")
if r.status_code in (404, 501, 503):
    raise RuntimeError("GCP module not active — run notebook against an env with the gcp_bucket extension enabled.")
assert r.status_code == 200, f"init-upload: {r.text}"

upload_uri = r.json()["upload_uri"]
upload_id = r.json()["upload_id"]
print(f"upload_id  : {upload_id}")
print(f"upload_uri : {upload_uri[:90]}…")


## Step 2 — PUT the file bytes

In [ ]:
# Step 2 — PUT bytes to the signed URL. Reuse the fixture so the schema
# matches what notebook 01 patched into items_schema.
from pathlib import Path
# Resolve relative to this notebook so the cell works regardless of cwd.
_NB_DIR = Path(globals().get("__vsc_ipynb_file__", os.getcwd())).parent
_FIXTURE = next(
    (p / "fixtures" / "sample.geojson" for p in [_NB_DIR, *_NB_DIR.parents]
     if (p / "fixtures" / "sample.geojson").exists()),
    Path("fixtures/sample.geojson"),
)
with open(_FIXTURE, "rb") as f:
    payload_bytes = f.read()

up_client = httpx.Client(timeout=120.0)
resp = up_client.put(upload_uri, content=payload_bytes, headers={"Content-Type": CONTENT_TYPE})
print(f"PUT signed URL: {resp.status_code}")
assert resp.status_code in (200, 201), f"signed PUT failed: {resp.text[:200]}"
print(f"uploaded {len(payload_bytes)} bytes")
up_client.close()


## Step 3 — Register the asset (pub/sub if remote, manual if local)

In [ ]:
# Step 3 — Asset registration:
# - REMOTE: the GCS OBJECT_FINALIZE pub/sub event fires asynchronously and
#   auto-registers the asset. Poll until the asset row exists.
# - LOCAL : no pub/sub; register manually so the ingestion step has an asset.
ASSET_URI = f"gs://{BUCKET}/{COLLECTION_ID}/{FILENAME}"

if PUBSUB_ENABLED:
    print("REMOTE_ONLY: waiting for pub/sub-driven asset auto-creation…")
    asset_url = f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/{ASSET_ID}"
    for attempt in range(30):
        r = client.get(asset_url)
        if r.status_code == 200:
            print(f"  asset materialized after {attempt+1} polls")
            break
        time.sleep(2)
    else:
        print("  WARN: asset did not appear within 60s; falling back to manual register")
        r = client.post(
            f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
            content=json.dumps({"asset_id": ASSET_ID, "asset_type": "ASSET", "uri": ASSET_URI, "metadata": {}}),
        )
        print(f"  manual register: {r.status_code}")
else:
    print("LOCAL: pub/sub disabled — registering asset manually.")
    r = client.post(
        f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
        content=json.dumps({"asset_id": ASSET_ID, "asset_type": "ASSET", "uri": ASSET_URI, "metadata": {}}),
    )
    print(f"manual register: {r.status_code} — {r.text[:200]}")
    assert r.status_code in (200, 201, 409), f"asset register: {r.text}"


## Step 4 — Run ingestion with GcsDetailedReporter (JSONL → catalog bucket)

In [ ]:
# Step 4 — Run ingestion with GcsDetailedReporter; minimum config only.
ingest_inputs = {
    "catalog_id": CATALOG_ID,
    "collection_id": COLLECTION_ID,
    "ingestion_request": {
        "asset": {"asset_id": ASSET_ID, "uri": ASSET_URI, "metadata": {}},
        "column_mapping": {
            "external_id": "asset_id",
            "attributes_source_type": "explicit_list",
            "attribute_mapping": [
                {"source": "name", "map_to": "name"},
                {"source": "datetime", "map_to": "datetime"},
            ],
        },
        "time_validity_start_column": "datetime",
        "source_srid": 4326,
        "encoding": "utf-8",
        "read_batch_size": 1000,
        "database_batch_size": 1000,
        "reporting": {
            "gcs_detailed_reporter": {
                "report_file_path": REPORT_PREFIX + "/{task_id}-{timestamp_utc}.jsonl",
                "output_format": "JSONL",
                "report_content": "ALL",
                "reported_fields": ["geoid", "asset_id", "name"],
            }
        },
    },
}

r = client.post(
    f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution",
    content=json.dumps({"inputs": ingest_inputs, "outputs": {}}),
    headers={**headers, "Prefer": "wait=true"},
)
print(f"ingestion: {r.status_code} — {r.text[:300]}")
assert r.status_code in (200, 201), f"ingestion exec: {r.text}"
job = r.json()
print(f"status={job.get('status')} jobID={job.get('jobID')}")


## Step 5 — Re-ingest: write policy refuses duplicate asset_id

In [ ]:
# Step 5 — Re-ingest the SAME asset → write policy refuses duplicate asset_id.
# The reporter writes a JSONLines line for each refused row.
r = client.post(
    f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution",
    content=json.dumps({"inputs": ingest_inputs, "outputs": {}}),
    headers={**headers, "Prefer": "wait=true"},
)
print(f"second ingestion: {r.status_code} — {r.text[:300]}")
assert r.status_code in (200, 201), f"second ingestion exec: {r.text}"
job2 = r.json()
print(f"status={job2.get('status')} jobID={job2.get('jobID')}")
print("(check the jsonlines under REPORT_PREFIX — refused rows are tagged with status='refused' and matcher='external_id')")


## Step 6 — List the JSONLines reports

In [ ]:
# Step 6 — List the JSONLines reports we just created.
# /gcp/buckets/.../objects exposes a listing endpoint; if not available, the
# user can list via gcloud / GCS console.
list_url = f"/gcp/buckets/catalogs/{CATALOG_ID}/objects"
r = client.get(list_url, params={"prefix": f"{COLLECTION_ID}/reports/"})
print(f"list status: {r.status_code}")
if r.status_code == 200:
    body = r.json()
    objects = body.get("objects") or body.get("items") or body
    if isinstance(objects, list):
        for obj in objects[:20]:
            print(" ", obj.get("name") or obj)
    else:
        print(json.dumps(body, indent=2)[:600])
else:
    print(f"  listing endpoint not available ({r.status_code}); use `gcloud storage ls gs://{BUCKET}/{COLLECTION_ID}/reports/` instead.")
client.close()
